# Решение для соревнования по классификации токсичных комментариев на русском языке


In [9]:
!pip install -q lightning torchmetrics transformers pandas scikit-learn numpy

Импорты и настройки

In [10]:
import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torchmetrics import AUROC
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import os

os.makedirs('logs', exist_ok=True)

SEED = 2026
L.seed_everything(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_float32_matmul_precision('high')

# Параметры токенезатор - максимально приближен к sota на русском языке
# При умеренном потреблении памяти
MODEL_NAME = 'sergeyzh/rubert-mini-frida'
MAX_LENGTH = 256
BATCH_SIZE = 32
# Лучший показатель, взят из эмпирических соображений
LEARNING_RATE = 3e-5

print(f"Устройство: {'GPU' if torch.cuda.is_available() else 'CPU'}")

INFO: Seed set to 2026
INFO:lightning.fabric.utilities.seed:Seed set to 2026


Устройство: GPU


In [11]:
class CommentDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]).strip() or "[ПУСТОЙ КОММЕНТАРИЙ]"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float32)
        return item


In [12]:
class ToxicClassifier(L.LightningModule):
    def __init__(self, model_name, learning_rate=5e-5, class_weight=None):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weight'])

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1,
            trust_remote_code=True
        )

        self.criterion = torch.nn.BCEWithLogitsLoss()
        self.class_weight = class_weight
        self.train_auc = AUROC(task='binary')
        self.val_auc = AUROC(task='binary')

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)

    def training_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        logits = self(input_ids, attention_mask)
        loss = self.criterion(logits, labels)

        # Веса для несбалансированных классов
        if self.class_weight:
            pos_mask = (labels > 0.5)
            weight = torch.ones_like(labels)
            weight[pos_mask] = self.class_weight
            loss = (loss * weight).mean()

        preds = torch.sigmoid(logits)
        self.train_auc.update(preds, labels.long())
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_auc', self.train_auc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        logits = self(input_ids, attention_mask)
        loss = self.criterion(logits, labels)
        preds = torch.sigmoid(logits)

        self.val_auc.update(preds, labels.long())
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_auc', self.val_auc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_auc'
            }
        }

 Загрузка данных

In [13]:
print("\nЗагружаем данные...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Подготовка данных
train_df['text'] = train_df['text'].fillna('').astype(str)
test_df['text'] = test_df['text'].fillna('').astype(str)

positive_ratio = train_df['score'].mean()
print(f"Данных для обучения: {len(train_df)}")
print(f"Токсичные комментарии: {positive_ratio:.1%}")

# Вес для редкого класса
pos_weight = (1 - positive_ratio) / positive_ratio if positive_ratio > 0 else 1.0
print(f"Вес токсичных комментариев: {pos_weight:.1f}")


Загружаем данные...
Данных для обучения: 173803
Токсичные комментарии: 18.0%
Вес токсичных комментариев: 4.6


Разделение на обучение и валидацию

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'].values,
    train_df['score'].values,
    test_size=0.1,
    random_state=SEED,
    stratify=(train_df['score'] > 0.5).astype(int)
)


Токенизация

In [15]:
print("\nЗагружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Создаем датасеты
train_dataset = CommentDataset(X_train, y_train, tokenizer, max_length=MAX_LENGTH)
val_dataset = CommentDataset(X_val, y_val, tokenizer, max_length=MAX_LENGTH)
test_dataset = CommentDataset(test_df['text'].values, tokenizer=tokenizer, max_length=MAX_LENGTH)

# Загрузчики данных
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, num_workers=2, pin_memory=True)

print(f"Обучение: {len(train_loader)} батчей")
print(f"Валидация: {len(val_loader)} батчей")


Загружаем токенизатор...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

Обучение: 4889 батчей
Валидация: 272 батчей


Обучение модели

In [16]:
print("\nНачинаем обучение...")
model = ToxicClassifier(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    class_weight=pos_weight
)

# Коллбэки
checkpoint = ModelCheckpoint(monitor='val_auc', mode='max', save_top_k=1)
# Ранняя остановка для предотвращения переобучения
early_stop = EarlyStopping(monitor='val_auc', patience=3, mode='max')

trainer = L.Trainer(
    max_epochs=5,
    accelerator='auto',
    callbacks=[checkpoint, early_stop],
    log_every_n_steps=10,
    gradient_clip_val=1.0,
    accumulate_grad_batches=2
)

trainer.fit(model, train_loader, val_loader)


Начинаем обучение...


config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/129M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sergeyzh/rubert-mini-frida and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ BertForSequenceClassification │ 32.3 M │ eval  │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss             │      0 │ train │     0 │
│ 2 │ train_auc │ BinaryAUROC                   │      0 │ train │     0 │
│ 3 │ val_auc   │ BinaryAUROC                   │      0 │ train │     0 │
└───┴───────────┴───────────────────────────────┴────────┴───────┴───────┘

Trainable params: 32.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 32.3 M                                                                                               
Total estimated model params size (MB): 129                                                                        
Modules in train mode: 3                                                                                           
Modules in eval mode: 141                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 141 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO: `Trainer.fit` stopped: `max_epochs=3` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


Предсказания и сохранение

In [17]:
print("\nДелаем предсказания...")
if checkpoint.best_model_path:
    model = ToxicClassifier.load_from_checkpoint(checkpoint.best_model_path)

model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        predictions.extend(probs)

# Сохраняем результаты
submission = pd.DataFrame({'id': test_df['id'], 'score': predictions})
submission.to_csv('submission.csv', index=False)

print(f"Сохранено: submission.csv")
print(f"Среднее значение: {np.mean(predictions):.4f}")



Делаем предсказания...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sergeyzh/rubert-mini-frida and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Сохранено: submission.csv
Среднее значение: 0.2064


Скачиваем файл в Colab

In [18]:

try:
    from google.colab import files
    files.download('submission.csv')
    print("Файл готов к скачиванию")
except:
    print("Результаты сохранены в submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Файл готов к скачиванию
